# 1. От фиксированных признаков к обучению ResNet
Цель: почувствовать, что модель — обычный `nn.Module`, состояние и градиенты которого
можно контролировать. Метрики и разбиение считаем знакомыми.

Пройдите TODO, затем Restart & Run All. Проверки диагностируют ошибки кода;
accuracy не обязана совпасть до последнего знака. Подсказки — в `birdlab/`,
полное решение — в `solutions/01_resnet.ipynb`.

In [ ]:
from pathlib import Path
import os, sys
# Start in repository root or its notebooks/solutions directory.
ROOT = Path.cwd()
if ROOT.name in {"notebooks", "solutions"}:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("BIRD_DATA", str(ROOT / "data/cub8")))
assert (DATA / "manifest.jsonl").exists(), "Run scripts/prepare_data.py first"
import torch
from torch import nn
torch.manual_seed(42)
torch.set_num_threads(4)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

In [ ]:
import json
from torch.utils.data import DataLoader
from birdlab.data import Birds, transforms, open_rgb
import matplotlib.pyplot as plt
classes = json.loads((DATA / "classes.json").read_text())
train_data = Birds(DATA, "train", transforms(True))
val_data = Birds(DATA, "val")
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, row in zip(axes.flat, train_data.rows[::max(1, len(train_data)//8)]):
    ax.imshow(open_rgb(DATA / row["image"]))
    ax.set_title(row["species"]); ax.axis("off")
plt.show()

## TODO 1 — свой модуль
Создайте `BirdClassifier`: ResNet-18 без финального fc и отдельная `Linear(512, K)`.
`set_stage('head')` замораживает backbone; `last_block` размораживает layer4.
Переопределите `train`: у замороженных BatchNorm running statistics не меняются.

Математика: $z=f_\theta(x)$, $\ell=Wz+b$. Какие параметры меняются в каждом режиме?
Чем `requires_grad=False` отличается от `eval()`?

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
class BirdClassifier(nn.Module):
    def __init__(self, n_classes, pretrained=True):
        super().__init__()
        raise NotImplementedError("TODO: backbone, head, initial stage")
    def set_stage(self, stage):
        raise NotImplementedError("TODO: select trainable parameters")
    def train(self, mode=True):
        raise NotImplementedError("TODO: preserve frozen BatchNorm state")
    def forward(self, images):
        raise NotImplementedError("TODO: features -> logits")

In [ ]:
probe = BirdClassifier(len(classes), pretrained=False)
probe.train()
old_mean = probe.backbone.bn1.running_mean.clone()
logits = probe(torch.randn(2, 3, 64, 64))
assert logits.shape == (2, len(classes))
logits.sum().backward()
assert probe.head.weight.grad is not None
assert probe.backbone.conv1.weight.grad is None
torch.testing.assert_close(old_mean, probe.backbone.bn1.running_mean)
probe.set_stage("last_block")
assert probe.backbone.layer4[0].conv1.weight.requires_grad
del probe

## TODO 2 — одна эпоха
Реализуйте `run_epoch(model, loader, device, optimizer=None)`. При наличии optimizer
обучаем, иначе оцениваем. Возвращаем средний по изображениям loss и accuracy.
Перед `cross_entropy` softmax не нужен. Почему?

In [ ]:
def run_epoch(model, loader, device, optimizer=None):
    raise NotImplementedError("TODO: forward, loss, backward, step, weighted metrics")

In [ ]:
tiny = nn.Linear(4, 2)
loader = [(torch.randn(8, 4), torch.zeros(8, dtype=torch.long))]
opt = torch.optim.SGD(tiny.parameters(), lr=.1)
before = run_epoch(tiny, loader, "cpu")["loss"]
for _ in range(20):
    run_epoch(tiny, loader, "cpu", opt)
assert run_epoch(tiny, loader, "cpu")["loss"] < before

## Эксперимент: голова и последний блок
Сначала 3 эпохи головы, затем 5 эпох последнего блока. Сохраняем лучший validation.
Сравните кривые; при желании замените голову на MLP и повторите с тем же seed.

In [ ]:
model = BirdClassifier(len(classes)).to(DEVICE)
loaders = {"train": DataLoader(train_data, batch_size=32, shuffle=True),
           "val": DataLoader(val_data, batch_size=32)}
optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3)
history, best = [], -1
RUN = ROOT / "runs/notebook_resnet"
RUN.mkdir(parents=True, exist_ok=True)
for epoch in range(8):
    if epoch == 3:
        model.set_stage("last_block")
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    train_score = run_epoch(model, loaders["train"], DEVICE, optimizer)
    val_score = run_epoch(model, loaders["val"], DEVICE)
    history.append((train_score["accuracy"], val_score["accuracy"]))
    print(epoch, train_score, val_score)
    if val_score["accuracy"] > best:
        best = val_score["accuracy"]
        torch.save({"state_dict": model.state_dict(), "classes": classes}, RUN / "model.pt")
plt.plot(history); plt.legend(["train", "validation"]); plt.show()

## Сохранение и применение
Загрузите лучший checkpoint в новый экземпляр. Объясните, почему вместе с весами
нужно сохранять порядок классов. После всех решений оцените на test один раз.

In [ ]:
checkpoint = torch.load(RUN / "model.pt", map_location=DEVICE, weights_only=True)
restored = BirdClassifier(len(classes), pretrained=False).to(DEVICE)
restored.load_state_dict(checkpoint["state_dict"])
test_loader = DataLoader(Birds(DATA, "test"), batch_size=32)
print(run_epoch(restored, test_loader, DEVICE))

## Бонус: кэш признаков
В режиме `head` вычислите и сохраните $z=f(x)$ через `torch.no_grad()`. Обучите на них
линейную голову и MLP. Измерьте скорость. Почему новый случайный crop на каждой эпохе
уже нельзя получить из одного вектора? Как это меняет сравнение с online-обучением?